In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine learning algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Display settings
pd.set_option("display.max_columns", None)

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Convert TotalCharges into numeric format
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Replace invalid values with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Check data types and missing values
print(df.info())

print("\nMissing values:")
print(df.isnull().sum().sum())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
# Target variable
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Input features
X = df.drop(
    columns=["customerID", "Churn"]
)

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Feature dataset shape:", X.shape)
print("\nNumerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

print("\nTarget distribution:")
print(y.value_counts())

Feature dataset shape: (7043, 19)

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training data shape: (5634, 19)
Testing data shape: (1409, 19)

Training target distribution:
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


In [6]:
# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [7]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "SVM": SVC(
        probability=True,
        random_state=42
    )
}

print("Machine learning models defined:")
for name in models:
    print("-", name)

Machine learning models defined:
- Logistic Regression
- Decision Tree
- Random Forest
- SVM


In [10]:

import xgboost as xgb
print("XGBoost version:", xgb.__version__)


XGBoost version: 3.4.1


In [11]:
models["XGBoost"] = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("All models defined:")
for name in models:
    print("-", name)

print("\nTotal models:", len(models))

All models defined:
- Logistic Regression
- Decision Tree
- Random Forest
- SVM
- XGBoost

Total models: 5


In [12]:
model_pipelines = {}

for name, model in models.items():
    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    model_pipelines[name] = model_pipeline

print("Complete model pipelines created:\n")

for name in model_pipelines:
    print("-", name)

Complete model pipelines created:

- Logistic Regression
- Decision Tree
- Random Forest
- SVM
- XGBoost


In [13]:
# Configure 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print("Cross-validation configured successfully!")
print("Number of folds:", cv.get_n_splits())
print("Evaluation metrics:", list(scoring.keys()))

Cross-validation configured successfully!
Number of folds: 5
Evaluation metrics: ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']


In [14]:
# Store cross-validation results
cv_results = {}

# Train and evaluate each model
for name, pipeline in model_pipelines.items():

    print(f"\nTraining: {name}")

    scores = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    cv_results[name] = scores

    print("Mean Accuracy:", round(scores["test_accuracy"].mean(), 4))
    print("Mean Precision:", round(scores["test_precision"].mean(), 4))
    print("Mean Recall:", round(scores["test_recall"].mean(), 4))
    print("Mean F1 Score:", round(scores["test_f1"].mean(), 4))
    print("Mean ROC-AUC:", round(scores["test_roc_auc"].mean(), 4))

print("\nCross-validation completed for all models!")


Training: Logistic Regression
Mean Accuracy: 0.8021
Mean Precision: 0.6529
Mean Recall: 0.5431
Mean F1 Score: 0.5923
Mean ROC-AUC: 0.8462

Training: Decision Tree
Mean Accuracy: 0.7267
Mean Precision: 0.4855
Mean Recall: 0.497
Mean F1 Score: 0.4909
Mean ROC-AUC: 0.6534

Training: Random Forest
Mean Accuracy: 0.7838
Mean Precision: 0.6222
Mean Recall: 0.4736
Mean F1 Score: 0.5374
Mean ROC-AUC: 0.8199

Training: SVM
Mean Accuracy: 0.8035
Mean Precision: 0.6772
Mean Recall: 0.4977
Mean F1 Score: 0.5735
Mean ROC-AUC: 0.7998

Training: XGBoost
Mean Accuracy: 0.8021
Mean Precision: 0.6623
Mean Recall: 0.5211
Mean F1 Score: 0.5828
Mean ROC-AUC: 0.8465

Cross-validation completed for all models!


In [15]:
# Create a detailed cross-validation summary
results_summary = []

for name, scores in cv_results.items():

    results_summary.append({
        "Model": name,

        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),

        "Precision Mean": scores["test_precision"].mean(),
        "Precision Std": scores["test_precision"].std(),

        "Recall Mean": scores["test_recall"].mean(),
        "Recall Std": scores["test_recall"].std(),

        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),

        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()
    })

results_df = pd.DataFrame(results_summary)

display(results_df.round(4))

,Model,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std
0,Logistic Regression,0.8021,0.0118,0.6529,0.0262,0.5431,0.0411,0.5923,0.0299,0.8462,0.0126
1,Decision Tree,0.7267,0.0074,0.4855,0.0139,0.4970,0.0228,0.4909,0.0151,0.6534,0.0106
2,Random Forest,0.7838,0.0128,0.6222,0.0330,0.4736,0.0296,0.5374,0.0278,0.8199,0.0126
3,SVM,0.8035,0.0088,0.6772,0.0276,0.4977,0.0157,0.5735,0.0162,0.7998,0.0158
4,XGBoost,0.8021,0.0130,0.6623,0.0360,0.5211,0.0290,0.5828,0.0270,0.8465,0.0093


In [16]:
# Import GridSearchCV to search for the best hyperparameter combination
from sklearn.model_selection import GridSearchCV


# Define the hyperparameter combinations we want to test
# classifier__ is used because our Logistic Regression is inside a Pipeline
param_grid_lr = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"],
    "classifier__solver": ["lbfgs"]
}


# Create GridSearchCV using our complete preprocessing and model pipeline
grid_lr = GridSearchCV(
    estimator=model_pipelines["Logistic Regression"],

    # Test every combination defined in param_grid_lr
    param_grid=param_grid_lr,

    # Select the best combination based on F1-score
    scoring="f1",

    # Use the same 5-fold stratified cross-validation
    cv=cv,

    # Use all available CPU cores
    n_jobs=-1,

    # Display progress during the search
    verbose=1
)


# Train and evaluate all parameter combinations
grid_lr.fit(X_train, y_train)


# Display the best hyperparameter combination found
print("\nBest Parameters:")
print(grid_lr.best_params_)


# Display the highest mean cross-validation F1-score
print("\nBest Cross-Validation F1 Score:")
print(round(grid_lr.best_score_, 4))

Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best Parameters:
{'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__solver': 'lbfgs'}

Best Cross-Validation F1 Score:
0.6295


In [17]:
# Define the hyperparameter combinations to test for XGBoost
# classifier__ refers to the XGBoost classifier inside our Pipeline
param_grid_xgb = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [3, 4],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__scale_pos_weight": [1, 2, 3]
}


# Create GridSearchCV using the complete XGBoost pipeline
grid_xgb = GridSearchCV(
    estimator=model_pipelines["XGBoost"],

    # Test all combinations in the parameter grid
    param_grid=param_grid_xgb,

    # Select the best configuration using F1-score
    scoring="f1",

    # Use five-fold stratified cross-validation
    cv=cv,

    # Use all available CPU cores
    n_jobs=-1,

    # Display search progress
    verbose=1
)


# Train XGBoost with every parameter combination
grid_xgb.fit(X_train, y_train)


# Display the best hyperparameters found
print("\nBest XGBoost Parameters:")
print(grid_xgb.best_params_)


# Display the best mean cross-validation F1-score
print("\nBest XGBoost Cross-Validation F1 Score:")
print(round(grid_xgb.best_score_, 4))

Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best XGBoost Parameters:
{'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 100, 'classifier__scale_pos_weight': 2}

Best XGBoost Cross-Validation F1 Score:
0.6367


In [18]:
# Define the hyperparameter combinations for SVM
# classifier__ refers to SVC inside our preprocessing Pipeline
param_grid_svm = {
    "classifier__C": [0.1, 1, 10],
    "classifier__gamma": ["scale", "auto"],
    "classifier__class_weight": [None, "balanced"]
}


# Create GridSearchCV using the complete SVM pipeline
grid_svm = GridSearchCV(
    estimator=model_pipelines["SVM"],

    # Test all combinations from the parameter grid
    param_grid=param_grid_svm,

    # Select the best parameter combination using F1-score
    scoring="f1",

    # Apply five-fold stratified cross-validation
    cv=cv,

    # Use all available CPU cores
    n_jobs=-1,

    # Display progress during the search
    verbose=1
)


# Train SVM using every hyperparameter combination
grid_svm.fit(X_train, y_train)


# Display the best hyperparameter combination
print("\nBest SVM Parameters:")
print(grid_svm.best_params_)


# Display the highest mean cross-validation F1-score
print("\nBest SVM Cross-Validation F1 Score:")
print(round(grid_svm.best_score_, 4))

Fitting 5 folds for each of 12 candidates, totalling 60 fits


d:\Rudra Prasad Dash\vs code\GitHub\CUSTEMER-CHURN-ML\venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Best SVM Parameters:
{'classifier__C': 10, 'classifier__class_weight': 'balanced', 'classifier__gamma': 'auto'}

Best SVM Cross-Validation F1 Score:
0.6256


In [20]:
# Collect the best F1-score obtained by each GridSearchCV experiment
tuned_results = {
    "Logistic Regression": grid_lr.best_score_,
    "XGBoost": grid_xgb.best_score_,
    "SVM": grid_svm.best_score_
}


# Convert the dictionary into a DataFrame for easier comparison
tuned_results_df = pd.DataFrame(
    list(tuned_results.items()),
    columns=["Model", "Best CV F1 Score"]
)


# Sort models from highest to lowest F1-score
tuned_results_df = tuned_results_df.sort_values(
    by="Best CV F1 Score",
    ascending=False
).reset_index(drop=True)


# Display the comparison table with four decimal places
display(tuned_results_df.round(4))


# Identify the model with the highest cross-validation F1-score
print(
    "\nCurrent leading model:",
    tuned_results_df.iloc[0]["Model"]
)

,Model,Best CV F1 Score
0,XGBoost,0.6367
1,Logistic Regression,0.6295
2,SVM,0.6256



Current leading model: XGBoost


In [21]:
# Store the best tuned pipelines returned by GridSearchCV
best_models = {
    "Logistic Regression": grid_lr.best_estimator_,
    "XGBoost": grid_xgb.best_estimator_,
    "SVM": grid_svm.best_estimator_
}


# Create a list to store evaluation results
tuned_cv_results = []


# Evaluate each tuned model using the same five-fold cross-validation
for name, model in best_models.items():

    print(f"Evaluating: {name}")

    # Calculate accuracy, precision, recall, F1 and ROC-AUC
    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    # Store the mean and standard deviation of each metric
    tuned_cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1 Score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })


# Convert the evaluation results into a DataFrame
tuned_cv_df = pd.DataFrame(tuned_cv_results)


# Sort models by F1-score, from highest to lowest
tuned_cv_df = tuned_cv_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)


# Display the final tuned-model comparison
display(tuned_cv_df.round(4))

Evaluating: Logistic Regression
Evaluating: XGBoost
Evaluating: SVM


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,XGBoost,0.7804,0.5677,0.7251,0.6367,0.8495
1,Logistic Regression,0.7508,0.5199,0.7980,0.6295,0.8455
2,SVM,0.7503,0.5193,0.7873,0.6256,0.8284
